# 11 Qwen Family Fine Tune

## Overview

This notebook runs one full family specific fine tune example for Qwen.
It shows the teaching checkpoint for this family, renders a sample conversation through the tokenizer, formats the dataset with the model's own chat template when available, and then runs a LoRA or QLoRA training path when the hardware is ready.

## Prerequisites

* Install the notebook dependencies in the next cell.
* Log in to Hugging Face first if the family is gated.
* Use CUDA for a real training run. CPU mode is for inspection only.
* Read [`MODEL_FAMILIES.md`](../MODEL_FAMILIES.md) before training if you have not reviewed the license or access rules yet.


In [ ]:
%pip install -q transformers==4.48.2 peft==0.10.0 trl==0.8.6 datasets==2.19.0 torch==2.2.2 accelerate==0.29.3 bitsandbytes==0.43.1


In [ ]:
from pathlib import Path
import sys

sys.path.append(str(Path("..").resolve()))

import torch
from datasets import Dataset
from peft import LoraConfig, PeftModel, get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, TrainingArguments
from trl import SFTTrainer

from utils.helpers import DATASETS_DIR, OUTPUT_DIR, format_chat_example, get_device, load_jsonl, print_trainable_parameters
from utils.model_families import get_model_family

profile = get_model_family("Qwen")
profile


## Family Profile And Tokenizer Preview

This cell shows the chosen teaching checkpoint, its access note, and one rendered prompt example.


In [ ]:
device = get_device()
model_name = profile.example_model_id
output_dir = OUTPUT_DIR / "qwen-adapter"
checkpoint_dir = OUTPUT_DIR / "qwen-checkpoints"

print(f"Device: {device}")
print(f"Teaching checkpoint: {model_name}")
print(f"License note: {profile.license_name}")
print(f"Access note: {profile.gated_access}")
print(f"Hardware note: {profile.full_example_hardware}")

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=profile.trust_remote_code,
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

messages = []
if profile.use_system_prompt:
    messages.append({"role": "system", "content": 'You are a helpful Qwen fine tuning tutor.'})
messages.append({"role": "user", "content": "Explain what clean fine tuning data should look like in two short sentences."})

if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
    rendered_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
else:
    rendered_prompt = "Tokenizer chat template not available. The notebook will fall back to the Alpaca style helper."

token_count = len(tokenizer(rendered_prompt)["input_ids"]) if rendered_prompt != "Tokenizer chat template not available. The notebook will fall back to the Alpaca style helper." else "fallback"
print(rendered_prompt)
print(f"Token count: {token_count}")


## Build A Family Specific Training Dataset

The key difference from the earlier generic notebooks is that this notebook formats the examples with the family tokenizer when possible.


In [ ]:
rows = load_jsonl(DATASETS_DIR / "sample_dataset.jsonl")
train_rows = rows[:8] if device == "cuda" else rows[:2]

formatted_records = [
    {
        "text": format_chat_example(
            tokenizer,
            item["instruction"],
            item["input"],
            item["output"],
            system_prompt='You are a helpful Qwen fine tuning tutor.' if profile.use_system_prompt else None,
        )
    }
    for item in train_rows
]

train_dataset = Dataset.from_list(formatted_records)
train_dataset[0]


## Load The Model And Attach LoRA

This uses a 4 bit load on CUDA so the teaching checkpoint is easier to fit on a normal GPU.


In [ ]:
if device == "cuda":
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
    )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        quantization_config=quant_config,
        device_map="auto",
        trust_remote_code=profile.trust_remote_code,
    )
    model = prepare_model_for_kbit_training(model)
else:
    model = None
    print("CPU mode detected. Model loading and training stay disabled in this notebook until you switch to CUDA.")

if model is not None:
    lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM",
)

    model = get_peft_model(model, lora_config)
    print_trainable_parameters(model)


In [ ]:
if model is not None:
    training_args = TrainingArguments(
        output_dir=str(checkpoint_dir),
        num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        learning_rate=2e-4,
        logging_steps=1,
        save_strategy="no",
        fp16=device == "cuda",
        report_to="none",
    )

    trainer = SFTTrainer(
        model=model,
        tokenizer=tokenizer,
        train_dataset=train_dataset,
        dataset_text_field="text",
        max_seq_length=256,
        args=training_args,
    )
else:
    trainer = None


## Run Training

CUDA runs perform a real training pass.
CPU runs stop before training.


In [ ]:
if trainer is not None:
    trainer.train()
    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"Saved adapter to {output_dir}")
else:
    print("CPU mode detected. This notebook stays in inspection mode until you move to CUDA or a hosted GPU runtime.")


## Compare Base And Adapter Behavior

Run this after a successful CUDA training pass.


In [ ]:
def generate_text(current_model, prompt: str) -> str:
    inputs = tokenizer(prompt, return_tensors="pt")
    if device == "cuda":
        inputs = {key: value.to(current_model.device) for key, value in inputs.items()}
    outputs = current_model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return text[len(prompt):].strip()

comparison_messages = []
if profile.use_system_prompt:
    comparison_messages.append({"role": "system", "content": 'You are a helpful Qwen fine tuning tutor.'})
comparison_messages.append({"role": "user", "content": "Explain why consistent output style matters in fine tuning."})

if hasattr(tokenizer, "apply_chat_template") and tokenizer.chat_template:
    comparison_prompt = tokenizer.apply_chat_template(
        comparison_messages,
        tokenize=False,
        add_generation_prompt=True,
    )
else:
    comparison_prompt = "### Instruction:\nExplain why consistent output style matters in fine tuning.\n\n### Response:\n"

if device != "cuda":
    print("Skip generation comparison on CPU. Use CUDA after the adapter is trained.")
elif output_dir.exists() and any(output_dir.iterdir()):
    clean_base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        trust_remote_code=profile.trust_remote_code,
    )
    adapted_base_model = AutoModelForCausalLM.from_pretrained(
        model_name,
        device_map="auto",
        trust_remote_code=profile.trust_remote_code,
    )
    adapter_loaded = PeftModel.from_pretrained(adapted_base_model, output_dir)

    base_response = generate_text(clean_base_model, comparison_prompt)
    adapted_response = generate_text(adapter_loaded, comparison_prompt)

    print("Base model response:\n")
    print(base_response)
    print("\nAdapter loaded response:\n")
    print(adapted_response)
else:
    print("No trained adapter found yet. Finish a CUDA run first, then rerun this comparison cell.")


## What To Watch In The Qwen Track

* The tokenizer is part of the training recipe.
* The license and access flow matter as much as the model size.
* The sample dataset is tiny on purpose. It teaches the workflow, not production quality.
* If you want a better family specific result, build a larger dataset in the style that matches this model family.
